In [1]:
from src.utils import get_data_env
from src.dataloading import EpiConfig, DataOrchestrator

disease_name    = 'measles'
nuts_level      = 'nuts2'
min_date        = '2012-01-01'
max_date        = '2023-12-31'
split_trainval  = '2017-06-01'
split_valtest   = '2018-06-01'
split_berlin    = False

horizon_size    = 1
horizon_leadtime= 1
sequence_length = 1
lag_num         = 1

config = EpiConfig(
    disease             = disease_name,
    data_env_dir        = get_data_env(),
    date_range          = (min_date, max_date),
    horizon_size        = horizon_size,
    sequence_length     = sequence_length,
    horizon_leadtime    = horizon_leadtime,
    lag_num             = lag_num,
    nuts_level          = nuts_level,
    # log_transform       = ['incidence','population_size'],
    split_berlin        = split_berlin,
    include_population  = False,
    split_trainval      = split_trainval, 
    split_valtest       = split_valtest,
    target_column       = 'cases',
    lag_column          = 'incidence',  
    verbose             = 0,
    prediction_mode     ='regression'
    )    

data_orchestrator = (DataOrchestrator(config)
                        .load_raw()
                        .harmonize_raw()
                        .process_data()
                        .build_features()
                        .normalize()
                        .finalize()
                        )

# bd = BaseLineDataLoaderManager(data_orchestrator)
# sd = ShallowDataLoaderManager(data_orchestrator).construct_dataloaders()
# gl = GraphDataLoaderManager(data_orchestrator).retrieve_static_graph('geographical_neighbors1').construct_dataloaders()

## AirpConfig

In [2]:
from src.dataloading.airp import AirpConfig, AirpOrchestrator

airpconfig                  = AirpConfig(data_orchestrator)
airporchestration           = AirpOrchestrator(airpconfig, data_orchestrator).build_orchestration()

AirpConfig loaded succesfully ✓


## Graph structure

In [ ]:
from src.utils.textformatting import align
import torch 
from dataclasses import dataclass
from typing import Optional, List

class GraphStructureError(Exception):
    def __init__(self, explanation: str):
        statement = "Error found in graphstructure!" + "\n" + explanation
        super().__init__(statement)

@dataclass
class NutsLayerGraphData:
    """
    Data for graph layer L1

    Parameters
    ----------
    x: torch.Tensor 
    edge_index: torch.Tensor 
    edge_weight: torch.Tensor 
    y: torch.Tensor
    """

    x:          torch.Tensor
    edge_index: torch.Tensor
    edge_weight:torch.Tensor
    y:          torch.Tensor
        
    def __post_init__(self):
        # validate input
        l, num_edges                        = self.edge_index.shape
        num_nodes_x, num_feat_x, num_seq_x  = self.x.shape
        num_nodes_y, num_horizon_y          = self.y.shape
        
        if l != 2:
            raise GraphStructureError('NutsLayerGraphData edge_index is expected to be of length 2')
        
        if num_edges != len(self.edge_weight):
            raise GraphStructureError(f'NutsLayerGraphData edge_index contains {num_edges} edges while edge_weight {len(self.edge_weight)} edges')     

        if num_nodes_x != num_nodes_y:
            raise GraphStructureError(f'NutsLayerGraphData x contains {num_nodes_x} nodes while y {num_nodes_y} nodes') 

        # TODO: Validate num_feat_x, num_seq_x, num_horizon_y against config                       
        
    def __repr__(self):
        cls = self.__class__.__name__
        info = (
            f"x={tuple(self.x.shape)}, "
            f"y={tuple(self.y.shape)}, "
            f"edge_index={tuple(self.edge_index.shape)}, "
            f"edge_weight={tuple(self.edge_weight.shape)}"
        )
        return f"{cls}({info})"
    
@dataclass
class AirportLayerGraphData:
    """
    Data for graph layer L2

    Parameters
    ----------
    x: torch.Tensor 
    edge_index: torch.Tensor 
    edge_weight: torch.Tensor
    """
    x:          torch.Tensor
    edge_index: torch.Tensor
    edge_weight:torch.Tensor
        
    def __repr__(self):
        cls = self.__class__.__name__
        info = (
            f"x={tuple(self.x.shape)}, "
            f"edge_index={tuple(self.edge_index.shape)}, "
            f"edge_weight={tuple(self.edge_weight.shape)}"
        )
        return f"{cls}({info})"    

    def __post_init__(self):
        # validate input
        l, num_edges                        = self.edge_index.shape
        num_nodes_x, num_feat_x, num_seq_x  = self.x.shape
        
        if l != 2:
            raise GraphStructureError('AirportLayerGraphData edge_index is expected to be of length 2')
        
        if num_edges != len(self.edge_weight):
            raise GraphStructureError(f'AirportLayerGraphData edge_index contains {num_edges} edges while edge_weight {len(self.edge_weight)} edges')     

        # TODO: Validate num_nodes_x, num_feat_x, num_seq_x, num_horizon_y against config of airport data
                
class NestedGraphData:
    """
    Data for graph layers L1 and L2 combined

    Parameters
    ----------
    nuts_layer:    NutsLayerGraphData
    airport_layer: AirportLayerGraphData
    """
    def __init__(self, 
                 nuts_layer:    NutsLayerGraphData, 
                 airport_layer: AirportLayerGraphData):
        
        # NUTS layer: L1
        self.x_l1           = nuts_layer.x 
        self.edge_index_l1  = nuts_layer.edge_index
        self.edge_weight_l1 = nuts_layer.edge_weight

        # Airport layer: L2
        self.x_l2           = airport_layer.x
        self.edge_index_l2  = airport_layer.edge_index 
        self.edge_weight_l2 = airport_layer.edge_weight 

        # Targets -> for L1
        self.y              = nuts_layer.y

        # based on airport graph
        num_airports      = self.edge_index_l2[0].max().item() + 1
        num_nodes_l2      = self.edge_index_l2[1].max().item() + 1       
        num_nodes_l1      = max(self.edge_index_l1[0].max().item(),self.edge_index_l1[1].max().item()) + 1

        if num_nodes_l1 != num_nodes_l2:
            raise GraphStructureError(f'num nodes found in L2: {num_nodes_l2} and in L1: {num_nodes_l1}')

    def to(self, device):
        """Move all tensors to device"""
        device = torch.device(device)   # Normalize device string/object
  
        self.x_l1           = self.x_l1.to(device)
        self.edge_index_l1  = self.edge_index_l1.to(device)
        self.edge_weight_l1 =self.edge_weight_l1.to(device)

        self.x_l2           = self.x_l2.to(device)  
        self.edge_index_l2  = self.edge_index_l2.to(device)
        self.edge_weight_l2 = self.edge_weight_l2.to(device)        

        self.y              = self.y.to(device)
        return self

    def cpu(self):
        """Convenience method to move to CPU"""
        return self.to('cpu')

    def cuda(self, device=None):
        """Convenience method to move to CUDA"""
        if device is None:
            return self.to('cuda')
        return self.to(f'cuda:{device}')

    @property
    def device(self):
        """Get current device of the data (from first layer's x tensor)"""
        return self.x_l1.device

    def __repr__(self):
        lines = f'<{self.__class__.__name__}(L1, L2)>'
        return lines

    def __str__(self):
        indent = 4
        keys   = ['edge_weight', 'edge_index']
        width  = max(len(k) for k in keys) + 1
        # Build output
        lines = [f'<{self.__class__.__name__}(']

        first_header = align('L1',"NUTS", 1)
        second_header = align('L2',"Airports", 1)

        lines.append(first_header)
        lines.append(" "*indent+"_"*len(first_header))
        lines.append(align('x', tuple(self.x_l1.shape), width))
        lines.append(align('edge_index', tuple(self.edge_index_l1.shape), width))        
        lines.append(align('edge_weight', tuple(self.edge_weight_l1.shape), width)) 
        lines.append('')         
        lines.append(second_header)     
        lines.append(" "*indent+"_"*len(second_header))          
        lines.append(align('x', tuple(self.x_l2.shape), width))
        lines.append(align('edge_index', tuple(self.edge_index_l2.shape), width))        
        lines.append(align('edge_weight', tuple(self.edge_weight_l2.shape), width))         
        lines.append(')>')
        
        return '\n'.join(lines)
    
class GraphDataSet:
    """
    A list of Data objects, for now limited to NestedGraphData objects

    Parameters
    ----------
    data_list: List[NestedGraphData]
    """    
    def __init__(self, data_list: List[NestedGraphData]):
        self.data_list  = data_list
        self._data_class= data_list[0].__class__.__name__

    def __iter__(self):
        return iter(self.data_list)
    
    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx: int):
        return self.data_list[idx]                   

    def __repr__(self):
        return self.__str__()

    def __str__(self) -> str:
        keys   = ['datapoints','dataclass']
        width  = max(len(k) for k in keys) + 1
        # Build output
        lines = [f'<{self.__class__.__name__}(\n']
        lines.append(align('datapoints',self.__len__(),width,newline=True))
        lines.append(align('dataclass',self._data_class, width))
        lines.append(')>')
        
        return ''.join(lines)